# 03 EEA Batch Ingestion

## Purpose
Load local EEA historical files, normalize pollutants, map stations to cities, validate rows, and aggregate daily city-level Silver data.

## Inputs
Local EEA CSV/Parquet files and the city/station mapping.

## Outputs
`data/silver/eea_city_daily.parquet` when explicitly written.

## Technologies used
Python, pandas, pyarrow, Jupyter.

## Configuration
No external EEA downloads. Historical EEA data is not mixed with Open-Meteo API data.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))


## Implementation
The EEA loader, station mapping, data quality, and aggregation logic are migrated from the legacy `src/ingestion/eea_loader.py` and station mapping module.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

CORE_POLLUTANTS = {"PM2.5", "PM10", "NO2"}
POLLUTANT_LABEL_MAP = {
    "PM2.5": "PM2.5", "PM2,5": "PM2.5", "Particles < 2.5 µm (aerodynamic diameter)": "PM2.5",
    "PM10": "PM10", "Particles < 10 µm (aerodynamic diameter)": "PM10",
    "NO2": "NO2", "Nitrogen dioxide": "NO2", "Nitrogen dioxide (air)": "NO2",
}
STATION_MAPPING = pd.DataFrame([
    {"city_id": "vienna_at", "eea_station_id": "AT90TAB", "mapping_status": "selected"},
    {"city_id": "berlin_de", "eea_station_id": "DEBE068", "mapping_status": "selected"},
    {"city_id": "paris_fr", "eea_station_id": "FR04143", "mapping_status": "selected"},
    {"city_id": "madrid_es", "eea_station_id": "ES0118A", "mapping_status": "selected"},
    {"city_id": "rome_it", "eea_station_id": "IT1906A", "mapping_status": "selected"},
    {"city_id": "amsterdam_nl", "eea_station_id": "NL00014", "mapping_status": "selected"},
    {"city_id": "warsaw_pl", "eea_station_id": "PL0592A", "mapping_status": "selected"},
    {"city_id": "prague_cz", "eea_station_id": "CZ0ARIE", "mapping_status": "selected"},
])

def _first_existing(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"missing one of {candidates}")

def load_eea_raw(path: Path | str) -> pd.DataFrame:
    path = Path(path)
    raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    station = _first_existing(raw, ["AirQualityStationEoICode", "AirQualityStation", "station_id"])
    ts = _first_existing(raw, ["DatetimeBegin", "datetime_begin", "Start", "date"])
    pollutant = _first_existing(raw, ["AirPollutant", "pollutant", "Pollutant", "Component"])
    value = _first_existing(raw, ["Concentration", "concentration", "Value", "value"])
    unit = _first_existing(raw, ["Unit", "unit"])
    df = pd.DataFrame({
        "eea_station_id": raw[station].astype(str).str.strip(),
        "datetime_begin": pd.to_datetime(raw[ts], utc=True, errors="coerce"),
        "pollutant": raw[pollutant].astype(str).str.strip().map(POLLUTANT_LABEL_MAP),
        "concentration": pd.to_numeric(raw[value], errors="coerce"),
        "unit": raw[unit].astype(str).str.strip(),
    })
    return df.dropna(subset=["datetime_begin", "pollutant", "concentration", "unit"]).query("concentration >= 0").reset_index(drop=True)

def map_stations_to_cities(raw_df: pd.DataFrame, mapping_df: pd.DataFrame = STATION_MAPPING) -> pd.DataFrame:
    selected = mapping_df.query("mapping_status == 'selected'")[["eea_station_id", "city_id"]].drop_duplicates()
    return raw_df.merge(selected, on="eea_station_id", how="inner")

def validate_eea_rows(df: pd.DataFrame) -> pd.DataFrame:
    required = ["city_id", "datetime_begin", "pollutant", "concentration", "unit"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required EEA row columns: {missing}")
    if df[required].isna().any().any():
        raise ValueError("required EEA fields must not be null")
    out = df.copy()
    out["datetime_begin"] = pd.to_datetime(out["datetime_begin"], utc=True, errors="coerce")
    if out["datetime_begin"].isna().any():
        raise ValueError("invalid datetime_begin")
    out = out[out["concentration"].ge(0) & out["pollutant"].isin(CORE_POLLUTANTS)].copy()
    return out

def aggregate_to_city_daily(mapped_df: pd.DataFrame, processing_time_utc=None) -> pd.DataFrame:
    df = validate_eea_rows(mapped_df)
    if processing_time_utc is None:
        processing_time_utc = datetime.now(timezone.utc)
    df["date"] = df["datetime_begin"].dt.date
    daily = df.groupby(["city_id", "date", "pollutant", "unit"], as_index=False).agg(
        mean_value=("concentration", "mean"),
        min_value=("concentration", "min"),
        max_value=("concentration", "max"),
        observation_count=("concentration", "count"),
    )
    daily["source"] = "eea"
    daily["processing_time_utc"] = pd.Timestamp(processing_time_utc)
    return daily

def write_eea_city_daily_parquet(df: pd.DataFrame, output_path: Path = Path("data/silver/eea_city_daily.parquet")) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)
    return output_path


## Validation / Quality Checks
Reject missing required fields, invalid dates, unsupported pollutants, and negative measurements. Verify aggregation math with a tiny in-memory sample.

In [ ]:
sample = pd.DataFrame([
    {"eea_station_id": "AT90TAB", "datetime_begin": "2023-01-15T08:00:00Z", "pollutant": "PM2.5", "concentration": 10.0, "unit": "µg/m³"},
    {"eea_station_id": "AT90TAB", "datetime_begin": "2023-01-15T09:00:00Z", "pollutant": "PM2.5", "concentration": 20.0, "unit": "µg/m³"},
])
mapped = map_stations_to_cities(sample)
daily = aggregate_to_city_daily(mapped, processing_time_utc=datetime(2026, 5, 30, tzinfo=timezone.utc))
assert daily.iloc[0]["mean_value"] == 15.0
assert daily.iloc[0]["observation_count"] == 2
print("EEA validation passed")


## Results
The notebook provides the historical file/batch source path and Silver daily output contract.

## Limitations
Station representativeness and local file availability remain project limitations.

## Next step
Run notebook `04` to collect contextual city metadata from Wikipedia.